# Uji penerimaan SLR_09 v0.8.3

Acuan FINAL083 berisi 50 usulan narasi sintetis yang masih PENDING. Kategori, tujuh field dan token harus direview manusia sebelum model dijalankan. Persetujuan lama tidak mengesahkan acuan ini. Kandidat dibekukan, cutoff dan target tetap. Notebook menolak template yang belum direview. Belum dieksekusi.

In [ ]:
from pathlib import Path
import os,sys,json,subprocess
from datetime import datetime,timezone
ROOT=Path(os.environ['SLR09_PACKAGE_ROOT']).resolve()
WORK=Path(os.environ['SLR09_WORK_DIR']).resolve()
REVIEW=Path(os.environ['SLR09_REVIEW_XLSX']).resolve()
SENA=Path(os.environ['SLR09_SENA_ARCHIVE']).resolve()
NARA=Path(os.environ['SLR09_NARA_ARCHIVE']).resolve()
sys.path.insert(0,str(ROOT/'code/acceptance'))
from import_review_xlsx import convert
from evaluate_acceptance import validate,write


In [ ]:
# Tolak kandidat/acuan lama sebelum impor atau pemuatan model.
template = json.loads((ROOT/'acceptance/review_bundle_draft.json').read_text())
if template['protocol']['binding']['candidate_version'] != '0.8.3':
    raise ValueError('Kandidat v0.8.3 belum siap; jangan menjalankan penerimaan dengan kandidat lama.')
if not all(row['id'].startswith('FINAL083-') for row in template['records']):
    raise ValueError('Notebook ini memerlukan acuan FINAL083 baru.')
# Penjagaan ini berjalan sebelum model dimuat atau acuan hasil impor disimpan.
bundle=convert(REVIEW,root=ROOT,validate_ready=True)
rows,preflight=validate(bundle,ROOT)
RUN=WORK/('final_acceptance_v083_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
RUN.mkdir(parents=True,exist_ok=False)
reviewed=RUN/'reviewed_reference.json'
write(reviewed,bundle)
print(json.dumps(preflight,ensure_ascii=False,indent=2))

# Simpan byte sumber yang terikat sebelum proses pemuatan/inferensi neural.
import shutil, hashlib
snapshot = RUN / 'pre_inference_snapshot'
snapshot_manifest = {}
for relative, expected in bundle['protocol']['binding']['files_sha256'].items():
    source = ROOT / relative
    target = snapshot / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    observed = hashlib.sha256(target.read_bytes()).hexdigest()
    assert observed == expected
    snapshot_manifest[relative] = observed
write(RUN/'pre_inference_snapshot_manifest.json', dict(files_sha256=snapshot_manifest, created_before_model_load=True))


In [ ]:
cmd=[sys.executable,str(ROOT/'code/acceptance/evaluate_acceptance.py'),'--root',str(ROOT),'--review',str(reviewed),'--output',str(RUN/'evidence'),'--sena-zip',str(SENA),'--nara-zip',str(NARA),'--device','cpu','--environment-label','CODEX_CPU_REPRESENTATIVE']
process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:print(line,end='',flush=True)
process.wait()
if process.returncode:raise RuntimeError('Evaluasi gagal; periksa workflow_result dan error_trace yang disimpan.')
report=json.loads((RUN/'evidence/evaluation.json').read_text())
print(json.dumps({k:report[k] for k in ['checks','metric_targets_met','decision','quality_acceptance_approved','card_done']},ensure_ascii=False,indent=2))


Jika gagal dan hasil dipakai untuk perbaikan, seluruh acuan yang telah diuji menjadi development. Siapkan acuan baru untuk putaran penerimaan berikutnya. Jangan mengubah target atau label setelah melihat prediksi untuk membuat putaran ini lulus.
